# CS224N · Lecture 4 — Dependency Parsing

**Slides:** [cs224n-spr2024-lecture04-dep-parsing.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture04-dep-parsing.pdf)  
**Prerequisites:** L2 (window classifiers, cross-entropy) and L3 (backprop, the forward/backward API).

### Key learnings
> **Explicit linguistic structure, and how a neural net can decide it.**

In Assignment 2 you build a neural dependency parser in PyTorch. This notebook builds one **from scratch** (NumPy, manual backprop), then shows the PyTorch version, then goes one step further to a **graph-based** parser.

| § | Topic | You will implement |
|---|---|---|
| 1 | Two views of syntax: constituency vs. dependency | A dependency-tree drawing tool |
| 2 | Why structure matters: attachment ambiguities | Catalan numbers; dependency paths for relation extraction |
| 3 | Dependency grammar, Universal Dependencies, treebanks | Reading CoNLL-U |
| 4 | Sources of information for parsing | Statistics from a treebank |
| 5 | Projectivity | A projectivity checker |
| 6 | Methods of dependency parsing | — |
| 7 | Greedy transition-based parsing (arc-standard) | The parser state machine + a training oracle |
| 8 | Evaluation: UAS and LAS | Scoring function |
| 9 | Handling non-projectivity | — |
| 10 | **Neural dependency parser** (Chen & Manning 2014) | Feature extraction, training, error analysis, ablation; PyTorch version |
| 11 | Further developments (SyntaxNet etc.) | — |
| 12 | **Graph-based parsing** (MST, Dozat & Manning 2017) | Chu-Liu/Edmonds + a neural arc-factored parser |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, itertools, collections, time, math

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)
random.seed(0)

def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

A small utility we'll use throughout: draw a dependency tree in the standard style, with words on a line and arrows from **head → dependent** above them (labeled with the relation).

In [ ]:
def draw_dependencies(words, heads, labels=None, title=None, ax=None, color="tab:blue"):
    # words: list of n tokens; heads[i] = head index of word i+1 (0 = ROOT); labels: relation names.
    toks = ["ROOT"] + list(words)
    n = len(words)
    if ax is None:
        _, ax = plt.subplots(figsize=(max(6, 1.1 * (n + 1)), 2.6))
    for i, w in enumerate(toks):
        ax.text(i, 0, w, ha="center", va="top", fontsize=11, weight="bold" if i == 0 else "normal")
    top = 1.0
    for d, h in enumerate(heads, start=1):
        if h is None:
            continue
        span = abs(h - d)
        height = 0.45 + 0.35 * span
        top = max(top, height)
        xs = np.linspace(h, d, 60)
        ys = 0.12 + height * (1 - ((xs - (h + d) / 2) / (span / 2)) ** 2)
        ax.plot(xs, ys, color=color, lw=1.2)
        ax.annotate("", xy=(d, 0.12), xytext=(xs[-4], ys[-4]),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=1.2))
        if labels is not None:
            ax.text((h + d) / 2, height + 0.18, labels[d - 1], ha="center", fontsize=8, color="darkred")
    ax.set_xlim(-0.8, n + 0.8); ax.set_ylim(-0.6, top + 0.6); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax

---
## 1. The linguistic structure of sentences: two views

### 1.1 Constituency = phrase structure grammar = context-free grammars (CFGs)
Phrase structure organizes words into **nested constituents**:
* Starting units: **words** (*the, cat, cuddly, by, door*), each with a category (Det, N, Adj, P).
* Words combine into **phrases**: *the cuddly cat* (NP), *by the door* (PP).
* Phrases combine into **bigger phrases**: *the cuddly cat by the door* (NP).

We can write this as CFG rules:
```
NP → Det (Adj)* N (PP)        e.g.  the cat,  a dog,  the large barking dog
PP → P NP                      e.g.  in a crate,  on the table,  by the door
VP → V PP                      e.g.  talk to the cat,  walked behind the dog
```

In [ ]:
# A constituency tree as nested tuples: (label, child1, child2, ...); leaves are strings.
tree = ("NP",
        ("NP", ("Det", "the"), ("Adj", "cuddly"), ("N", "cat")),
        ("PP", ("P", "by"), ("NP", ("Det", "the"), ("N", "door"))))

def bracketed(t):
    if isinstance(t, str):
        return t
    return "[" + t[0] + " " + " ".join(bracketed(c) for c in t[1:]) + "]"

def pretty(t, indent=0):
    if isinstance(t[1], str):
        print("  " * indent + f"{t[0]}: {t[1]}")
        return
    print("  " * indent + t[0])
    for c in t[1:]:
        pretty(c, indent + 1)

print(bracketed(tree), "\n")
pretty(tree)

### 1.2 Dependency structure
Dependency structure shows **which words depend on (modify, attach to, or are arguments of) which other words**. There are no phrase nodes, only word-to-word arrows.

*Look in the large crate in the kitchen by the door*: *crate* depends on *Look*; *in*, *the*, *large*, *kitchen* and *door* depend on *crate*; etc.

In [ ]:
words = "Look in the large crate in the kitchen by the door".split()
#         Look in  the large crate in the kitchen by the door
heads  = [0,    5,  5,  5,    1,    8, 8,  5,      11, 11, 5]
labels = ["root", "case", "det", "amod", "obl", "case", "det", "nmod", "case", "det", "nmod"]
draw_dependencies(words, heads, labels, title="A dependency tree"); plt.show()

---
## 2. Why is sentence structure needed for communication?

Humans communicate complex ideas by **composing words into bigger units** to convey complex meanings. Listeners need to work out **what modifies (attaches to) what**. A model needs to understand sentence structure to interpret language correctly, and language is full of structural ambiguity.

### 2.1 Prepositional phrase (PP) attachment ambiguity
**"Scientists count whales from space"** (a real headline):
* ✅ *from space* modifies **count**: the counting is done from space (by satellite).
* ❌ *from space* modifies **whales**: whales that come from space.

In [ ]:
w = "Scientists count whales from space".split()
fig, axes = plt.subplots(1, 2, figsize=(13, 2.8))
draw_dependencies(w, [2, 0, 2, 5, 2], ["nsubj", "root", "obj", "case", "obl"], ax=axes[0],
                  title="✓ counting from space (space → count)", color="tab:green")
draw_dependencies(w, [2, 0, 2, 5, 3], ["nsubj", "root", "obj", "case", "nmod"], ax=axes[1],
                  title="✗ whales from space (space → whales)", color="tab:red")
plt.show()

### 2.2 PP attachment ambiguities multiply
A key parsing decision is how we "attach" various constituents: PPs, adverbial or participial phrases, infinitives, coordinations, etc. *"The board approved [its acquisition] [by Royal Trustco Ltd.] [of Toronto] [for \$27 a share] [at its monthly meeting]."* Each PP could attach to several earlier words.

The number of possible structures grows as the **Catalan numbers**:
$$C_n = \frac{(2n)!}{(n+1)!\,n!}$$
an exponentially growing series that arises in many tree-like contexts (e.g., the number of triangulations of a polygon with $n+2$ sides; it also turns up in triangulating probabilistic graphical models). This is why parsers can't just enumerate all trees.

In [ ]:
catalan = lambda n: math.comb(2 * n, n) // (n + 1)
print("n  : " + " ".join(f"{n:>7}" for n in range(1, 13)))
print("C_n: " + " ".join(f"{catalan(n):>7}" for n in range(1, 13)))

### 2.3 Other ambiguities
* **Coordination scope:** *"Shuttle veteran and longtime NASA executive Fred Gregory appointed to board"*. Is that one person (a shuttle veteran who is also a NASA executive) or two people (a shuttle veteran, and Fred Gregory)? It depends on whether *veteran* and *executive* are conjoined as modifiers of *Fred Gregory*, or whether *veteran* is its own conjunct.
* **Adjectival/adverbial modifier ambiguity:** *"small business owner"*: is the business small ([small business] owner) or the owner small (small [business owner])?
* **Verb phrase (VP) attachment:** *"Mutilated body washes up on Rio beach to be used for Olympics beach volleyball"*. What attaches to what: is the *beach* to be used for volleyball, or the *body*?

### 2.4 Dependency paths help extract semantic interpretation
A simple practical example: **extracting protein–protein interactions** from biomedical text (Erkan et al. 2007; Fundel et al. 2007). In

*"The results demonstrated that KaiC interacts rhythmically with SasA, KaiA, and KaiB"*

the **dependency path** between two protein names reveals the relation, even though they're far apart in the word sequence:
* KaiC ←nsubj— interacts —nmod:with→ SasA
* KaiC ←nsubj— interacts —nmod:with→ SasA —conj:and→ KaiB

In [ ]:
w = "The results demonstrated that KaiC interacts rhythmically with SasA , KaiA and KaiB".split()
#    The results demonstrated that KaiC interacts rhythmically with SasA ,  KaiA and KaiB
h = [2,  3,      0,           6,   6,   3,        6,           9,   6,   11, 9,   13, 9]
l = ["det", "nsubj", "root", "mark", "nsubj", "ccomp", "advmod", "case", "nmod:with",
     "punct", "conj:and", "cc", "conj:and"]
draw_dependencies(w, h, l); plt.show()

def dependency_path(words, heads, labels, a, b):
    # Shortest path between words a and b (1-based) in the tree, with arrow directions.
    def ancestors(i):
        path = [i]
        while heads[path[-1] - 1] != 0:
            path.append(heads[path[-1] - 1])
        return path
    up_a, up_b = ancestors(a), ancestors(b)
    common = next(x for x in up_a if x in up_b)
    left, right = up_a[:up_a.index(common) + 1], up_b[:up_b.index(common)]
    out = words[a - 1]
    for i in left[1:]:
        out += f" <-{labels[left[left.index(i) - 1] - 1]}- {words[i - 1]}"
    for i in reversed(right):
        out += f" -{labels[i - 1]}-> {words[i - 1]}"
    return out

print(dependency_path(w, h, l, w.index("KaiC") + 1, w.index("SasA") + 1))
print(dependency_path(w, h, l, w.index("KaiC") + 1, w.index("KaiB") + 1))

---
## 3. Dependency grammar and dependency structure

**Dependency syntax** postulates that syntactic structure consists of **relations between lexical items**, normally binary asymmetric relations ("arrows") called **dependencies**.
* The arrows are commonly **typed** with the name of grammatical relations (subject, prepositional object, apposition, etc.).
* An arrow connects a **head** (governor, superior, regent) with a **dependent** (modifier, inferior, subordinate).
* Usually, dependencies form a **tree**: a connected, acyclic, single-root graph.
* We usually add a fake **ROOT** node, so every word is a dependent of precisely one other node.
* Some people draw arrows one way, some the other! Tesnière had them point from head to dependent, and we follow that convention.

### 3.1 History (brief)
The idea is old: **Pāṇini's grammar** of Sanskrit (c. 5th century BCE) and 1st-millennium Arabic grammarians used dependency-like analyses. Constituency/CFG is a "new-fangled" 20th-century invention (Wells 1947; Chomsky 1953). Modern dependency work is usually traced to **Lucien Tesnière (1959)**. It suits languages with freer word order and rich inflection (Russian, Latin). David Hays built an early (the first?) dependency parser in 1962.

### 3.2 The rise of annotated data: treebanks and Universal Dependencies
History: Brown corpus (1967; POS-tagged 1979) → Lancaster-IBM Treebank (late 1980s) → **Penn Treebank** (Marcus et al. 1993) → **Universal Dependencies (UD)**, a consistent annotation scheme for 100+ languages (universaldependencies.org).

Building a treebank first seems slower and less useful than writing a grammar by hand. But a treebank gives us:
* **reusability of the labor**: many parsers, POS taggers, etc. can be built on it; a valuable resource for linguistics
* **broad coverage**, not just a few intuitions
* **frequencies and distributional information** (which is what ML needs)
* **a way to evaluate NLP systems**

UD treebanks are distributed in the **CoNLL-U** format: one token per line with 10 tab-separated columns: `ID FORM LEMMA UPOS XPOS FEATS HEAD DEPREL DEPS MISC`. Here is the slides' example sentence, *"Bills on ports and immigration were submitted by Senator Brownback, Republican of Kansas"*, in CoNLL-U (unused columns are `_`):

In [ ]:
conllu = '''
# text = Bills on ports and immigration were submitted by Senator Brownback , Republican of Kansas
1   Bills        bill        NOUN   _ _ 7  nsubj:pass _ _
2   on           on          ADP    _ _ 3  case       _ _
3   ports        port        NOUN   _ _ 1  nmod       _ _
4   and          and         CCONJ  _ _ 5  cc         _ _
5   immigration  immigration NOUN   _ _ 3  conj       _ _
6   were         be          AUX    _ _ 7  aux:pass   _ _
7   submitted    submit      VERB   _ _ 0  root       _ _
8   by           by          ADP    _ _ 9  case       _ _
9   Senator      Senator     PROPN  _ _ 7  obl        _ _
10  Brownback    Brownback   PROPN  _ _ 9  flat       _ _
11  ,            ,           PUNCT  _ _ 12 punct      _ _
12  Republican   Republican  PROPN  _ _ 9  appos      _ _
13  of           of          ADP    _ _ 14 case       _ _
14  Kansas       Kansas      PROPN  _ _ 12 nmod       _ _
'''

def read_conllu(text):
    # Minimal CoNLL-U reader -> list of sentences, each a list of dicts. (Real files are tab-separated.)
    sents, cur = [], []
    for line in text.strip().split("\n") + [""]:
        line = line.strip()
        if not line:
            if cur:
                sents.append(cur); cur = []
        elif not line.startswith("#"):
            f = line.split()
            if "-" in f[0] or "." in f[0]:        # skip multiword-token and empty-node lines
                continue
            cur.append({"id": int(f[0]), "form": f[1], "lemma": f[2], "upos": f[3],
                        "head": int(f[6]), "deprel": f[7]})
    return sents

sent = read_conllu(conllu)[0]
for t in sent[:4]:
    print(t)
draw_dependencies([t["form"] for t in sent], [t["head"] for t in sent], [t["deprel"] for t in sent]); plt.show()

### 3.3 The most common UD relations (cheat sheet)
| Relation | Meaning | Example (head → dependent) |
|---|---|---|
| `nsubj` | nominal subject | *ate → I* |
| `obj` | direct object | *ate → fish* |
| `iobj` | indirect object | *gave → me* (gave me a book) |
| `obl` | oblique nominal (PP-like argument/adjunct of a verb) | *count → space* (from space) |
| `nmod` | nominal modifier of a noun | *whales → space* (whales from space) |
| `amod` | adjectival modifier | *crate → large* |
| `det` | determiner | *crate → the* |
| `case` | case marker / preposition (attached to its **noun**!) | *space → from* |
| `advmod` | adverbial modifier | *interacts → rhythmically* |
| `aux`, `aux:pass` | auxiliary verb | *submitted → were* |
| `ccomp`, `xcomp` | clausal complements | *demonstrated → interacts* |
| `mark` | subordinating marker | *interacts → that* |
| `conj`, `cc` | conjunct, coordinating conjunction | *ports → immigration*, *immigration → and* |
| `compound`, `flat` | multiword expressions / names | *Senator → Brownback* |
| `punct` | punctuation | *submitted → .* |
| `root` | ROOT → main predicate | *ROOT → submitted* |

UD puts **content words as heads**: prepositions hang off their noun via `case` and auxiliaries off the main verb. This makes trees more parallel across languages (many languages use case endings instead of prepositions).

---
## 4. A synthetic treebank, and the sources of information for parsing

We need data to train parsers on. Real treebanks (e.g., UD English-EWT) need a download, so to keep everything self-contained and fast, we generate a **synthetic treebank** from a small grammar with UD-style annotations. It contains the key phenomenon of §2, **PP-attachment ambiguity resolved by word choice**:
* *with + instrument* (telescope, spoon, brush, fork) → attaches to the **verb** (`obl`): *saw the man [with a telescope]*
* *with + attribute* (hat, beard, stripes, cover) → attaches to the **noun** (`nmod`): *saw the man [with a hat]*
* *in/near + location* → attaches to the verb (`obl`)

The POS sequence is identical in both *with* cases (`ADP DET NOUN`), so **only the words themselves** can tell the parser which attachment is right.

In [ ]:
DET = ["the", "a"]
ADJ = ["big", "small", "old", "red", "happy"]
AGENT = ["man", "woman", "dog", "cat", "child", "scientist", "chef"]
THING = ["ball", "book", "cake", "whale", "letter", "picture"]
VERB = ["saw", "ate", "read", "painted", "found", "counted"]
INSTR = ["telescope", "spoon", "brush", "fork"]     # 'with X' -> attaches to the VERB (obl)
ATTR = ["hat", "beard", "stripes", "cover"]          # 'with X' -> attaches to the NOUN (nmod)
LOC = ["park", "kitchen", "garden", "library"]       # 'in/near X' -> attaches to the VERB (obl)
ADV = ["quickly", "yesterday", "slowly"]

def gen_sentence():
    # Returns a list of (word, upos, head, deprel) tuples; heads are 1-based, 0 = ROOT.
    toks = []
    def add(w, pos):
        toks.append([w, pos, None, None]); return len(toks)
    def attach(i, head, rel):
        toks[i - 1][2:] = [head, rel]
    def NP(nouns, adj_p=0.4):
        d = add(random.choice(DET), "DET")
        adjs = []
        while random.random() < adj_p and len(adjs) < 2:
            adjs.append(add(random.choice(ADJ), "ADJ"))
        n = add(random.choice(nouns), "NOUN")
        attach(d, n, "det")
        for a in adjs:
            attach(a, n, "amod")
        return n
    def PP(prep, nouns, adj_p=0.15):
        p = add(prep, "ADP"); n = NP(nouns, adj_p); attach(p, n, "case")
        return n
    subj = NP(AGENT)
    if random.random() < 0.2:
        attach(PP("with", ATTR), subj, "nmod")             # the man with a hat ...
    v = add(random.choice(VERB), "VERB"); attach(v, 0, "root"); attach(subj, v, "nsubj")
    obj = NP(THING + AGENT); attach(obj, v, "obj")
    if random.random() < 0.3:
        attach(PP("with", ATTR), obj, "nmod")              # noun-attached PP first (keeps trees projective)
    if random.random() < 0.4:
        attach(PP("with", INSTR), v, "obl")                # instrument -> verb
    if random.random() < 0.4:
        attach(PP(random.choice(["in", "near"]), LOC), v, "obl")
    if random.random() < 0.3:
        attach(add(random.choice(ADV), "ADV"), v, "advmod")
    attach(add(".", "PUNCT"), v, "punct")
    for i in range(len(toks) - 1):                          # a -> an before vowels
        if toks[i][0] == "a" and toks[i + 1][0][0] in "aeiou":
            toks[i][0] = "an"
    return [tuple(t) for t in toks]

random.seed(0)
treebank = [gen_sentence() for _ in range(3000)]
train_sents, dev_sents = treebank[:2500], treebank[2500:]
print("average sentence length:", np.mean([len(s) for s in treebank]).round(1))
ex = treebank[0]
print(" ".join(t[0] for t in ex))
draw_dependencies([t[0] for t in ex], [t[2] for t in ex], [t[3] for t in ex]); plt.show()

### 4.1 What are the straightforward sources of information for dependency parsing?
1. **Bilexical affinities**: the dependency [*discussion* → *issues*] is plausible. (In our data: [*saw* → *telescope*] vs. [*man* → *hat*].)
2. **Dependency distance**: most (but not all) dependencies are between nearby words.
3. **Intervening material**: dependencies rarely span intervening verbs or punctuation.
4. **Valency of heads**: how many dependents, on which side, are usual for a head?

Example: *ROOT Discussion of the outstanding issues was completed .*

Let's measure these from our treebank (this is precisely the kind of "frequencies and distributional information" a treebank gives you):

In [ ]:
deps = [(i, t) for s in treebank for i, t in enumerate(s, 1) if t[2] != 0]
dist = collections.Counter(min(abs(i - t[2]), 8) for i, t in deps)
print("2. dependency distance |head - dep| (8 = 8+):")
for k in sorted(dist):
    print(f"   {k}: {dist[k] / len(deps):6.1%} " + "#" * int(60 * dist[k] / len(deps)))

left, right, count = collections.Counter(), collections.Counter(), collections.Counter()
for s in treebank:
    for i, t in enumerate(s, 1):
        count[t[1]] += 1
        for j, u in enumerate(s, 1):
            if u[2] == i:
                (left if j < i else right)[t[1]] += 1
print("\n4. valency: average number of left / right dependents per head POS")
for pos in ["VERB", "NOUN", "ADP", "DET"]:
    print(f"   {pos:5s}: left {left[pos] / count[pos]:.2f}   right {right[pos] / count[pos]:.2f}")

attach = collections.defaultdict(collections.Counter)
for s in treebank:
    for i, t in enumerate(s, 1):
        if t[3] in ("obl", "nmod") and any(u[0] == "with" and u[2] == i for u in s):
            attach[t[0]][s[t[2] - 1][1]] += 1
print("\n1. bilexical affinity: what does 'with X' attach to?")
for noun in ["telescope", "spoon", "hat", "beard"]:
    c = attach[noun]
    print(f"   with {noun:9s}: head is VERB {c['VERB'] / sum(c.values()):.0%}, NOUN {c['NOUN'] / sum(c.values()):.0%}")

---
## 5. Dependency parsing: constraints and projectivity

A sentence is parsed by **choosing for each word what other word (including ROOT) it is a dependent of**. Usually some constraints:
* Only one word is a dependent of ROOT.
* No cycles ($A\to B$, $B\to A$).

This makes the dependencies a **tree**. The final issue is whether arrows can **cross** (be **non-projective**) or not.

**Definition of a projective parse:** there are no crossing dependency arcs when the words are laid out in their linear order, with all arcs above the words.
* Dependencies corresponding to a CFG tree must be projective (forming dependencies by taking 1 child of each category as its head).
* Most syntactic structure is projective like this, but dependency theory normally does allow non-projective structures to account for **displaced constituents**. You can't easily get the semantics of certain constructions right without them.

Example: **"Who did Bill buy the coffee from yesterday?"** *from* belongs with *Who* (it's "from whom"), but *Who* has been moved to the front of the question, so the arc *Who → from* crosses the arc *buy → yesterday*.

In [ ]:
def is_projective(heads):
    # heads[i] = head of word i+1 (0 = ROOT). True iff no two arcs cross.
    arcs = [(min(h, d), max(h, d)) for d, h in enumerate(heads, start=1)]
    for (a, b), (c, e) in itertools.combinations(arcs, 2):
        if a < c < b < e or c < a < e < b:
            return False
    return True

w = "Who did Bill buy the coffee from yesterday ?".split()
h = [4, 4, 4, 0, 6, 4, 1, 4, 4]
l = ["obl", "aux", "nsubj", "root", "det", "obj", "case", "obl:tmod", "punct"]
draw_dependencies(w, h, l, title="Non-projective: Who→from crosses buy→yesterday"); plt.show()
print("projective?", is_projective(h))
print("all synthetic treebank trees projective?", all(is_projective([t[2] for t in s]) for s in treebank))

---
## 6. Methods of dependency parsing

1. **Dynamic programming.** Eisner (1996) gives a clever $O(n^3)$ algorithm by producing parse items with heads at the ends rather than in the middle. (Projective trees only.)
2. **Graph algorithms.** Create a **Minimum (or Maximum) Spanning Tree** for the sentence. McDonald et al.'s (2005) $O(n^2)$ **MSTParser** scores dependencies independently using an ML classifier (MIRA for online learning, but it can be anything). The neural graph-based parser of **Dozat & Manning (2017)** and its successors are very successful. → §12
3. **Constraint satisfaction.** Edges that don't satisfy hard constraints are eliminated (Karlsson 1990, etc.).
4. **"Transition-based parsing"** or "deterministic dependency parsing." Greedy choice of attachments guided by good machine-learning classifiers, e.g., **MaltParser** (Nivre et al. 2008). It has proven highly effective, and fast. → §7, §10

---
## 7. Greedy transition-based parsing (Nivre 2003)

A simple form of greedy **discriminative** dependency parser. The parser does a sequence of bottom-up actions, roughly like "shift" or "reduce" in a shift-reduce parser (as in compilers), but the "reduce" actions are specialized to create dependencies with the head on the left or the right.

The parser has:
* a **stack** $\sigma$, written with the top to the right, which starts with the ROOT symbol
* a **buffer** $\beta$, written with the top to the left, which starts with the input sentence
* a set of **dependency arcs** $A$, which starts off empty
* a set of **actions**

### 7.1 The arc-standard transition system
Start: $\sigma = [\text{ROOT}]$, $\beta = w_1,\dots,w_n$, $A = \emptyset$

| Action | Before | After |
|---|---|---|
| **Shift** | $\sigma,\ w_i\vert\beta,\ A$ | $\sigma\vert w_i,\ \beta,\ A$ |
| **Left-Arc$_r$** | $\sigma\vert w_i\vert w_j,\ \beta,\ A$ | $\sigma\vert w_j,\ \beta,\ A\cup\{r(w_j, w_i)\}$ |
| **Right-Arc$_r$** | $\sigma\vert w_i\vert w_j,\ \beta,\ A$ | $\sigma\vert w_i,\ \beta,\ A\cup\{r(w_i, w_j)\}$ |

Finish: $\sigma = [w]$ (just ROOT, in our implementation), $\beta = \emptyset$.

In words: **Shift** moves the next word onto the stack. **Left-Arc** makes the top word the head of the second word, and pops the second. **Right-Arc** makes the second word the head of the top word, and pops the top. A sentence of $n$ words always takes exactly $2n$ transitions ($n$ shifts, $n$ arcs), so parsing is **linear time**. (There are other transition schemes, e.g., arc-eager.)

In [ ]:
LABELS = sorted({t[3] for s in treebank for t in s})
TRANSITIONS = ["S"] + [f"L-{r}" for r in LABELS] + [f"R-{r}" for r in LABELS]
print(f"{len(LABELS)} labels -> {len(TRANSITIONS)} transitions (2|R| + 1):", LABELS)

class Configuration:
    def __init__(self, n):
        self.stack = [0]                      # 0 = ROOT
        self.buffer = list(range(1, n + 1))
        self.head, self.label = {}, {}        # the arc set A, as dep -> head / dep -> label

    def apply(self, t):
        if t == "S":
            self.stack.append(self.buffer.pop(0))
        elif t.startswith("L-"):              # second-from-top becomes a dependent of top
            dep = self.stack.pop(-2)
            self.head[dep], self.label[dep] = self.stack[-1], t[2:]
        else:                                 # top becomes a dependent of second-from-top
            dep = self.stack.pop()
            self.head[dep], self.label[dep] = self.stack[-1], t[2:]

    def legal(self):
        # Boolean mask over TRANSITIONS. ROOT can't be a dependent, and gets exactly one child at the very end.
        can_shift = len(self.buffer) > 0
        can_left = len(self.stack) >= 2 and self.stack[-2] != 0
        can_right = len(self.stack) >= 2 and (self.stack[-2] != 0 or len(self.buffer) == 0)
        return np.array([can_shift] + [can_left] * len(LABELS) + [can_right] * len(LABELS))

    def done(self):
        return not self.buffer and self.stack == [0]

    def children(self, h):
        return sorted(d for d, hh in self.head.items() if hh == h)

### 7.2 Which action to take? The oracle
To *train* a parser, we need the correct action sequence for each gold tree. A **static oracle** for arc-standard: given gold heads $h^*$ and the top two stack items $s_1$ (top) and $s_2$:
1. If $h^*(s_2) = s_1$ → **Left-Arc** (with $s_2$'s gold label).
2. Else if $h^*(s_1) = s_2$ **and all of $s_1$'s gold dependents are already attached** → **Right-Arc**. (The second condition matters: once $s_1$ is popped it can never get more dependents.)
3. Else → **Shift**.

Let's trace the slide example, **"I ate fish"**:

In [ ]:
def oracle(c, gold_head, gold_label):
    if len(c.stack) >= 2:
        s1, s2 = c.stack[-1], c.stack[-2]
        if s2 != 0 and gold_head[s2] == s1:
            return "L-" + gold_label[s2]
        if gold_head[s1] == s2 and all(d in c.head for d in gold_head if gold_head[d] == s1):
            return "R-" + gold_label[s1]
    return "S"

def show(c, words):
    name = lambda i: "[root]" if i == 0 else words[i - 1]
    return " ".join(name(i) for i in c.stack).ljust(22) + " | " + " ".join(name(i) for i in c.buffer)

words = ["I", "ate", "fish"]
gold_head, gold_label = {1: 2, 2: 0, 3: 2}, {1: "nsubj", 2: "root", 3: "obj"}
c = Configuration(len(words))
print(f"{'action':10s} {'stack':22s} | buffer");  print(f"{'start':10s} {show(c, words)}")
while not c.done():
    t = oracle(c, gold_head, gold_label)
    c.apply(t)
    new = [d for d in c.head if d not in getattr(c, "_seen", set())]
    c._seen = set(c.head)
    arc = "" if not new else f"   A += {c.label[new[0]]}({'[root]' if c.head[new[0]] == 0 else words[c.head[new[0]] - 1]} -> {words[new[0] - 1]})"
    print(f"{t:10s} {show(c, words)}{arc}")

In this example, at each step we chose the correct next transition because the oracle knows the answer. **A parser has to work this out: by exploring or inferring.** That's the machine-learning part.

Before training anything, verify the oracle on every tree in the treebank: following it must reproduce each gold tree exactly. This test is worth writing for any transition system you implement (it catches non-projective trees and oracle bugs).

In [ ]:
def oracle_sequence(sent):
    gh = {i: t[2] for i, t in enumerate(sent, 1)}
    gl = {i: t[3] for i, t in enumerate(sent, 1)}
    c, seq = Configuration(len(sent)), []
    while not c.done():
        t = oracle(c, gh, gl)
        seq.append((c, t))
        nxt = Configuration(0); nxt.stack, nxt.buffer = c.stack[:], c.buffer[:]
        nxt.head, nxt.label = dict(c.head), dict(c.label)
        nxt.apply(t); c = nxt
    assert c.head == gh and c.label == gl, "oracle failed to reconstruct the gold tree"
    return seq

n_trans = sum(len(oracle_sequence(s)) for s in treebank)
print(f"oracle reproduces all {len(treebank)} gold trees; {n_trans:,} transitions = 2 x {n_trans // 2:,} tokens")

### 7.3 MaltParser (Nivre & Hall 2005): choosing the action with ML
"We have left to explain how we choose the next action 🤷. Answer: stand back, I know machine learning!"
* Each action is predicted by a **discriminative classifier** (e.g., a softmax classifier) over each **legal** move: at most 3 untyped choices, at most $|R|\times2+1$ when typed.
* **Features:** top-of-stack word and POS; first-in-buffer word and POS; etc.
* There is **no search** (in the simplest form). But you can profitably do a **beam search** (slower but better): keep the $k$ best parse prefixes at each time step.
* Accuracy is a bit below the state of the art, but it gives **very fast linear-time parsing** with high accuracy, great for parsing the web.

### 7.4 Conventional feature representation
Feature templates combine 1–3 elements from the configuration, giving **binary, sparse indicator features** with dimension $10^6$–$10^7$:

In [ ]:
def indicator_features(c, sent):
    # Classic hand-designed feature templates (a small subset), as strings.
    W = lambda i: "ROOT" if i == 0 else sent[i - 1][0]
    P = lambda i: "ROOT" if i == 0 else sent[i - 1][1]
    s1 = c.stack[-1] if c.stack else None
    s2 = c.stack[-2] if len(c.stack) > 1 else None
    b1 = c.buffer[0] if c.buffer else None
    f = []
    if s1 is not None: f += [f"s1.w={W(s1)}", f"s1.t={P(s1)}", f"s1.wt={W(s1)}_{P(s1)}"]
    if b1 is not None: f += [f"b1.w={W(b1)}", f"b1.t={P(b1)}"]
    if s1 is not None and b1 is not None: f += [f"s1.w_b1.w={W(s1)}_{W(b1)}", f"s1.t_b1.t={P(s1)}_{P(b1)}"]
    if s2 is not None and s1 is not None: f += [f"s2.t_s1.t={P(s2)}_{P(s1)}", f"s2.w_s1.w={W(s2)}_{W(s1)}"]
    if s1 is not None:
        lc = [d for d in c.children(s1) if d < s1]
        if lc: f.append(f"lc1(s1).l={c.label[lc[0]]}")
    return f

sent0 = treebank[0]
cfg, _ = oracle_sequence(sent0)[8]
print("configuration:", show(cfg, [t[0] for t in sent0]))
for feat in indicator_features(cfg, sent0):
    print("  ", feat)

all_feats = set()
for s in treebank:
    for cc, _ in oracle_sequence(s):
        all_feats.update(indicator_features(cc, s))
print(f"\ndistinct indicator features on our tiny synthetic treebank: {len(all_feats):,}"
      " (real treebanks with full template sets: millions)")

---
## 8. Evaluation of dependency parsing: (labeled) dependency accuracy

$$\text{Acc} = \frac{\#\text{ correct deps}}{\#\text{ of deps}}$$
* **UAS** (unlabeled attachment score): fraction of words with the correct **head**.
* **LAS** (labeled attachment score): fraction with the correct head **and** label.

The slides' example, *"ROOT She saw the video lecture"*:

| idx | word | gold head | gold label | parsed head | parsed label |
|---|---|---|---|---|---|
| 1 | She | 2 | nsubj | 2 | nsubj ✓✓ |
| 2 | saw | 0 | root | 0 | root ✓✓ |
| 3 | the | 5 | det | **4** ✗ | det |
| 4 | video | 5 | nn | 5 ✓ | **nsubj** ✗ |
| 5 | lecture | 2 | obj | 2 ✓ | **ccomp** ✗ |

In [ ]:
def attachment_scores(gold, pred):
    # gold, pred: lists of (head, label) per word. Returns (UAS, LAS).
    uas = np.mean([g[0] == p[0] for g, p in zip(gold, pred)])
    las = np.mean([g == p for g, p in zip(gold, pred)])
    return uas, las

gold = [(2, "nsubj"), (0, "root"), (5, "det"), (5, "nn"), (2, "obj")]
pred = [(2, "nsubj"), (0, "root"), (4, "det"), (5, "nsubj"), (2, "ccomp")]
uas, las = attachment_scores(gold, pred)
print(f"UAS = {uas:.0%}   LAS = {las:.0%}   (slide: UAS 4/5 = 80%, LAS 2/5 = 40%)")

---
## 9. Handling non-projectivity

The arc-standard algorithm only builds **projective** trees. Possible directions:
1. Just declare defeat on non-projective arcs 🤷 (they're rare in English, but much more common in, e.g., Czech, Dutch, or German).
2. Use a dependency formalism that only has projective representations (a CFG only allows projective structures; you promote the head of projectivity violations).
3. Use a postprocessor on a projective parser's output to identify and resolve non-projective links (*pseudo-projective parsing*, Nivre & Nilsson 2005).
4. Add extra transitions that can model at least most non-projective structures. E.g., an extra **SWAP** transition that reorders the stack allows **any** non-projectivity (cf. bubble sort; Nivre 2009).
5. Move to a parsing mechanism that doesn't use or require any constraints on projectivity, e.g., the graph-based MSTParser or Dozat & Manning (2017). → §12

---
## 10. A neural dependency parser (Chen & Manning 2014)

### 10.1 Why do we gain from a neural dependency parser? Indicator features revisited
Indicator features have three problems:
* **Problem #1: sparse.** Most feature combinations are rarely seen, so their weights are poorly estimated.
* **Problem #2: incomplete.** Unseen word/tag combinations get no information at all; you can't hand-write every useful conjunction.
* **Problem #3: expensive to compute.** More than **95% of parsing time** is consumed by feature computation (string building + hashing into huge weight vectors).

**Neural approach: learn a dense and compact feature representation** (~1000 dimensions instead of $10^6$–$10^7$ sparse ones).

Results on English parsing to Stanford Dependencies:

| Parser | UAS | LAS | sentences/s |
|---|---|---|---|
| MaltParser | 89.8 | 87.2 | 469 |
| MSTParser | 91.4 | 88.1 | 10 |
| TurboParser | 92.3 | 89.6 | 8 |
| **C & M 2014** | **92.0** | **89.7** | **654** |

As accurate as the best graph-based parsers of the time, while **faster than the greedy MaltParser**.

### 10.2 First win: distributed representations
* Each word is a $d$-dimensional dense vector (a word embedding). Similar words get close vectors.
* **Part-of-speech tags and dependency labels are also represented as $d$-dimensional vectors.** The smaller discrete sets also have semantic similarities: NNS (plural noun) should be close to NN (singular noun); `nummod` (numerical modifier) should be close to `amod` (adjective modifier).

### 10.3 Extracting tokens and vector representations from a configuration
We extract a set of tokens based on stack/buffer positions (exactly the Chen & Manning feature set, $18 + 18 + 12 = 48$ tokens):
* **words and POS** ($18$ each): $s_1,s_2,s_3$, $b_1,b_2,b_3$; for $s_1$ and $s_2$: the leftmost/rightmost children $lc_1, rc_1$, the second leftmost/rightmost $lc_2, rc_2$, and the leftmost-of-leftmost $lc_1(lc_1)$ and rightmost-of-rightmost $rc_1(rc_1)$
* **arc labels** ($12$): the labels of those 12 children

Missing positions get a special `<NULL>` token. A **concatenation of the vector representations of all these** is the neural representation of the configuration. Here's the slide's example configuration (stack *[ROOT has good]*, buffer *[control .]*, arc *has → He* (nsubj)):

In [ ]:
NULL, ROOTTOK, UNK = "<NULL>", "<ROOT>", "<UNK>"

def feature_positions(c):
    # Return the 18 token positions (6 stack/buffer + 12 children); None = missing.
    st = [c.stack[-k] if len(c.stack) >= k else None for k in (1, 2, 3)]
    bu = [c.buffer[k] if len(c.buffer) > k else None for k in (0, 1, 2)]
    def lc(i, k):
        if i is None: return None
        ch = [d for d in c.children(i) if d < i]
        return ch[k - 1] if len(ch) >= k else None
    def rc(i, k):
        if i is None: return None
        ch = [d for d in c.children(i) if d > i][::-1]
        return ch[k - 1] if len(ch) >= k else None
    kids = []
    for s in st[:2]:
        kids += [lc(s, 1), rc(s, 1), lc(s, 2), rc(s, 2), lc(lc(s, 1), 1), rc(rc(s, 1), 1)]
    return st + bu, kids

def feature_strings(c, sent):
    base, kids = feature_positions(c)
    W = lambda i: NULL if i is None else (ROOTTOK if i == 0 else sent[i - 1][0])
    P = lambda i: NULL if i is None else (ROOTTOK if i == 0 else sent[i - 1][1])
    Lb = lambda i: NULL if i is None else c.label.get(i, NULL)
    toks = base + kids
    return [W(i) for i in toks], [P(i) for i in toks], [Lb(i) for i in kids]

NAMES = ["s1", "s2", "s3", "b1", "b2", "b3"] + [f"{f}({s})" for s in ("s1", "s2")
         for f in ("lc1", "rc1", "lc2", "rc2", "lc1.lc1", "rc1.rc1")]

demo = [("He", "PRP"), ("has", "VBZ"), ("good", "JJ"), ("control", "NN"), (".", ".")]
c = Configuration(len(demo))
c.stack, c.buffer = [0, 2, 3], [4, 5]
c.head, c.label = {1: 2}, {1: "nsubj"}
ws, ps, ls = feature_strings(c, demo)
print(f"{'position':12s} {'word':9s} {'POS':7s} label")
for k, name in enumerate(NAMES):
    lab = ls[k - 6] if k >= 6 else ""
    if ws[k] != NULL or k < 6:
        print(f"{name:12s} {ws[k]:9s} {ps[k]:7s} {lab}")
print("(all other children positions are <NULL>)")

In [ ]:
# Vocabularies (from training data only) and integer feature vectors.
w_vocab = [NULL, ROOTTOK, UNK] + sorted({t[0] for s in train_sents for t in s})
p_vocab = [NULL, ROOTTOK] + sorted({t[1] for s in treebank for t in s})
l_vocab = [NULL] + LABELS
W2I, P2I, L2I = ({w: i for i, w in enumerate(v)} for v in (w_vocab, p_vocab, l_vocab))

def features(c, sent):
    ws, ps, ls = feature_strings(c, sent)
    return [W2I.get(w, W2I[UNK]) for w in ws] + [P2I[p] for p in ps] + [L2I[l] for l in ls]

def make_training_data(sents):
    X, Y = [], []
    for s in sents:
        for c, t in oracle_sequence(s):
            X.append(features(c, s)); Y.append(TRANSITIONS.index(t))
    return np.array(X), np.array(Y)

X_train, Y_train = make_training_data(train_sents)
print("training examples (configurations):", X_train.shape, "  vocab sizes: words", len(w_vocab),
      "POS", len(p_vocab), "labels", len(l_vocab))
print("most common transitions:", [(TRANSITIONS[k], v) for k, v in collections.Counter(Y_train).most_common(4)])

### 10.4 Second win: deep learning classifiers are non-linear
Traditional ML classifiers (Naïve Bayes, SVMs, logistic regression, softmax) only give **linear decision boundaries**. (Indicator-feature parsers compensate with hand-made feature *conjunctions*.) Neural networks use multiple layers to learn much more complex **non-linear** decision boundaries, so the conjunctions are learned automatically.

### 10.5 The model architecture: a simple feed-forward multi-class classifier
$$x = [\,E^w_{w_1};\dots;E^w_{w_{18}};\ E^t_{t_1};\dots;E^t_{t_{18}};\ E^l_{l_1};\dots;E^l_{l_{12}}\,] \qquad\text{(lookup + concat)}$$
$$h = \text{ReLU}(Wx + b_1)$$
$$y = \text{softmax}(Uh + b_2)\quad\text{over } \{\text{Shift},\ \text{Left-Arc}_r,\ \text{Right-Arc}_r\}$$

The **log loss (cross-entropy) is backpropagated all the way to the embeddings.** The hidden layer re-represents the input (it moves inputs around in an intermediate vector space) so it can be easily classified with a (linear) softmax.

🔎 Details from the paper: $d=50$, hidden size 200, embeddings initialized from pretrained word vectors, dropout, AdaGrad, and a **cube** activation $h=(Wx+b)^3$, chosen because it directly models products of 3 features, like the classic 3-element feature templates. (ReLU is the standard choice now, and is what Assignment 2 uses.) The paper also **precomputes** $W$ times each embedding for the most frequent words, which is a big part of its speed.

Below: the forward pass, the backward pass (using L3's results: $\hat y - y$ at the logits, ReLU routes the gradient, $\delta x^\top$ for weights, and scatter-adding into embedding rows), and Adam.

In [ ]:
D_EMB, HIDDEN, N_T = 32, 200, len(TRANSITIONS)

def init_parser(seed=0):
    r = np.random.default_rng(seed)
    return {"Ew": r.normal(scale=0.1, size=(len(w_vocab), D_EMB)),
            "Ep": r.normal(scale=0.1, size=(len(p_vocab), D_EMB)),
            "El": r.normal(scale=0.1, size=(len(l_vocab), D_EMB)),
            "W": r.normal(scale=np.sqrt(2 / (48 * D_EMB)), size=(HIDDEN, 48 * D_EMB)),   # He init
            "b1": np.zeros(HIDDEN),
            "U": r.normal(scale=np.sqrt(1 / HIDDEN), size=(N_T, HIDDEN)),
            "b2": np.zeros(N_T)}

BLOCKS = [("Ew", slice(0, 18)), ("Ep", slice(18, 36)), ("El", slice(36, 48))]

def parser_forward(p, F):
    B = len(F)
    x = np.concatenate([p[name][F[:, cols]].reshape(B, -1) for name, cols in BLOCKS], axis=1)  # (B, 48d)
    z = x @ p["W"].T + p["b1"]
    h = np.maximum(z, 0)
    return x, z, h, h @ p["U"].T + p["b2"]

def parser_loss_and_grads(p, F, y):
    B = len(F)
    x, z, h, logits = parser_forward(p, F)
    P = softmax(logits, axis=1)
    loss = -np.log(P[np.arange(B), y] + 1e-12).mean()
    G = P; G[np.arange(B), y] -= 1; G /= B              # dJ/dlogits = (y_hat - y)/B
    g = {"U": G.T @ h, "b2": G.sum(0)}
    dz = (G @ p["U"]) * (z > 0)                          # back through ReLU
    g["W"], g["b1"] = dz.T @ x, dz.sum(0)
    dx = dz @ p["W"]                                     # (B, 48d) gradient for the embeddings
    off = 0
    for name, cols in BLOCKS:
        n_tok = cols.stop - cols.start
        g[name] = np.zeros_like(p[name])
        np.add.at(g[name], F[:, cols].reshape(-1), dx[:, off:off + n_tok * D_EMB].reshape(-1, D_EMB))
        off += n_tok * D_EMB
    return loss, g

class Adam:
    def __init__(self, params, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
    def step(self, params, grads):
        self.t += 1
        for k in params:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            m_hat, v_hat = self.m[k] / (1 - self.b1 ** self.t), self.v[k] / (1 - self.b2 ** self.t)
            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

In [ ]:
# Gradient check on a tiny batch (Lecture 3 habit: verify before training!).
p_chk = init_parser(seed=1)
Fb, yb = X_train[:4], Y_train[:4]
_, g_chk = parser_loss_and_grads(p_chk, Fb, yb)
for name, idx in [("U", (3, 5)), ("W", (7, 100)), ("Ew", (Fb[0, 0], 2)), ("El", (Fb[0, 40], 1))]:
    old = p_chk[name][idx]
    p_chk[name][idx] = old + 1e-5; lp, _ = parser_loss_and_grads(p_chk, Fb, yb)
    p_chk[name][idx] = old - 1e-5; lm, _ = parser_loss_and_grads(p_chk, Fb, yb)
    p_chk[name][idx] = old
    print(f"d loss/d {name}{idx}: analytic {g_chk[name][idx]: .6e}   numerical {(lp - lm) / 2e-5: .6e}")

In [ ]:
def parse(p, sent, use_words=True):
    # Greedy parsing: at each step take the highest-scoring LEGAL transition.
    c = Configuration(len(sent))
    while not c.done():
        f = np.array([features(c, sent)])
        if not use_words:
            f[:, :18] = W2I[NULL]
        logits = parser_forward(p, f)[3][0]
        logits[~c.legal()] = -np.inf
        c.apply(TRANSITIONS[int(np.argmax(logits))])
    return c

def evaluate(parse_fn, sents):
    gold, pred, errors = [], [], []
    for s in sents:
        heads, labels = parse_fn(s)
        for i, t in enumerate(s, 1):
            gold.append((t[2], t[3])); pred.append((heads[i], labels[i]))
            if heads[i] != t[2]:
                errors.append((s, i, heads[i]))
    uas, las = attachment_scores(gold, pred)
    return uas, las, errors

def train_parser(X, Y, epochs=3, use_words=True, batch_size=256, seed=0):
    X = X.copy()
    if not use_words:
        X[:, :18] = W2I[NULL]                    # ablation: hide all word identities
    p = init_parser(seed)
    opt, r = Adam(p), np.random.default_rng(seed)
    for ep in range(epochs):
        perm, total = r.permutation(len(X)), 0.0
        for k in range(0, len(X), batch_size):
            idx = perm[k:k + batch_size]
            loss, g = parser_loss_and_grads(p, X[idx], Y[idx])
            opt.step(p, g)
            total += loss * len(idx)
        print(f"  epoch {ep + 1}: mean loss {total / len(X):.4f}")
    return p

t0 = time.time()
parser = train_parser(X_train, Y_train)
def transition_parse(s):
    c = parse(parser, s); return c.head, c.label
uas, las, errors = evaluate(transition_parse, dev_sents[:300])
print(f"trained in {time.time() - t0:.0f}s.  Held-out UAS = {uas:.2%}   LAS = {las:.2%}")

### 10.6 Error analysis: what does the parser still get wrong?
High accuracy on its own tells you little. **Always look at the errors.**

In [ ]:
def pp_has_adjective(s, i):
    # Is there an adjective between the preposition and the noun i?
    j = i - 1
    while j >= 1 and s[j - 1][1] != "ADP":
        if s[j - 1][1] == "ADJ":
            return True
        j -= 1
    return False

by_rel = collections.Counter(s[i - 1][3] for s, i, _ in errors)
print(f"{len(errors)} attachment errors; by gold relation: {dict(by_rel)}")
print(f"errors on a PP noun whose PP contains an adjective: {sum(pp_has_adjective(s, i) for s, i, _ in errors)}/{len(errors)}\n")
for s, i, ph in errors[:6]:
    name = lambda k: "ROOT" if k == 0 else s[k - 1][0]
    print(f"  {' '.join(t[0] for t in s)}\n      '{s[i - 1][0]}': gold head '{name(s[i - 1][2])}', predicted '{name(ph)}'")

The remaining errors are all PP attachments, and every one has an **adjective inside the PP**: *"… the dog **with the small beard**"*. Why? The decision that matters is made when the object noun (*dog*) is on top of the stack and the buffer starts with *with*: should we finish the noun (Right-Arc it to the verb, after which the PP can only attach to the verb) or keep it to receive the PP (Shift)? The parser only sees $b_1, b_2, b_3$ = *with, the, small*. **The noun that decides the attachment (*beard* vs. *spoon*) is $b_4$: outside the feature window**, so the parser falls back on the more common verb attachment. A greedy parser must commit before it sees the evidence. That's a fundamental limitation of greedy transition-based parsing with fixed local features, and exactly what **beam search** (§11), **bidirectional sequence encoders** (L5–6), and **graph-based parsing** (§12) address.

### 10.7 Ablation: how much do the words matter? (bilexical affinities)
Retrain using only POS tags and labels (all 18 word features set to `<NULL>`):

In [ ]:
parser_pos = train_parser(X_train, Y_train, use_words=False)
def pos_only_parse(s):
    c = parse(parser_pos, s, use_words=False); return c.head, c.label
uas_p, las_p, errors_p = evaluate(pos_only_parse, dev_sents[:300])
print(f"\nwith words: UAS {uas:.2%}  |  POS only: UAS {uas_p:.2%}")
print("POS-only errors by gold relation:", dict(collections.Counter(s[i - 1][3] for s, i, _ in errors_p)))
print("most frequent POS-only error words:", collections.Counter(s[i - 1][0] for s, i, _ in errors_p).most_common(6))

Without word identities, the parser can't distinguish *with a telescope* from *with a hat*, or *in the park* from *with a hat*: all are `ADP DET NOUN`. It learns the more frequent attachment (to the verb), so every noun-attached PP (*hat, beard, stripes, cover*) comes out wrong. Since this is the only ambiguity in our grammar, every new error is a PP attachment. This is **source of information #1, bilexical affinities**, demonstrated: lexical information is essential for attachment decisions. (In real data, POS-only parsers lose several UAS points for the same reason.)

### 10.8 The same parser in PyTorch (Assignment 2 style)
This is the idiomatic version. It uses the same feature extraction, configurations, and data. Notice how much shorter the model is when autograd handles the backward pass, and the use of dropout (as in the paper).

In [ ]:
# [PyTorch] Chen & Manning-style feed-forward transition classifier.
import torch
import torch.nn as nn
import torch.nn.functional as F

class ParserModel(nn.Module):
    def __init__(self, n_words, n_pos, n_labels, n_trans, d=32, hidden=200, dropout=0.3):
        super().__init__()
        self.word_emb = nn.Embedding(n_words, d)
        self.pos_emb = nn.Embedding(n_pos, d)
        self.label_emb = nn.Embedding(n_labels, d)
        self.hidden = nn.Linear(48 * d, hidden)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(hidden, n_trans)

    def forward(self, f):                              # f: (B, 48) long
        x = torch.cat([self.word_emb(f[:, :18]).flatten(1),
                       self.pos_emb(f[:, 18:36]).flatten(1),
                       self.label_emb(f[:, 36:]).flatten(1)], dim=1)
        return self.out(self.dropout(F.relu(self.hidden(x))))   # logits

torch.manual_seed(0)
model = ParserModel(len(w_vocab), len(p_vocab), len(l_vocab), len(TRANSITIONS))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
Xt, Yt = torch.tensor(X_train), torch.tensor(Y_train)
for epoch in range(3):
    model.train()
    perm, total = torch.randperm(len(Xt)), 0.0
    for k in range(0, len(Xt), 256):
        idx = perm[k:k + 256]
        loss = F.cross_entropy(model(Xt[idx]), Yt[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(idx)
    print(f"epoch {epoch + 1}: loss {total / len(Xt):.4f}")

def torch_parse(sent):
    model.eval()
    c = Configuration(len(sent))
    with torch.no_grad():
        while not c.done():
            logits = model(torch.tensor([features(c, sent)]))[0]
            logits[~torch.tensor(c.legal())] = -float("inf")
            c.apply(TRANSITIONS[int(logits.argmax())])
    return c.head, c.label

uas_t, las_t, _ = evaluate(torch_parse, dev_sents[:300])
print(f"PyTorch parser: UAS {uas_t:.2%}  LAS {las_t:.2%}")

---
## 11. Further developments in transition-based neural parsing

The work was further developed by others, in particular at Google:
* bigger, deeper networks with better-tuned hyperparameters
* **beam search**
* **global, CRF-style inference** over the decision sequence (normalize over whole transition sequences, not each local decision)

This led to **SyntaxNet** and the **Parsey McParseface** model (2016), "The World's Most Accurate Parser":

| Method | UAS | LAS (PTB WSJ SD 3.3) |
|---|---|---|
| Chen & Manning 2014 | 92.0 | 89.7 |
| Weiss et al. 2015 | 93.99 | 92.05 |
| Andor et al. 2016 | 94.61 | 92.79 |

---
## 12. Graph-based dependency parsers

* **Compute a score for every possible dependency** (every choice of head) for each word. E.g., for "big" in *ROOT The big cat sat*: score(ROOT→big) = 0.5, score(The→big) = 0.3, score(cat→big) = 2.0, score(sat→big) = 0.8.
* Doing this well requires more than just knowing the two words: we need good **"contextual" representations** of each word token (developed in the coming lectures).
* Repeat for every word, then **find the best parse**: the highest-scoring spanning tree (**MST algorithm**).
* There are $n^2$ possible dependencies in a sentence of length $n$, so it's slower than transition-based parsing.

### 12.1 Finding the best tree: the Chu-Liu/Edmonds algorithm
Taking each word's best head independently may create **cycles**. Chu-Liu/Edmonds (1965/1967) finds the maximum spanning arborescence rooted at ROOT:
1. For each word, pick its highest-scoring head.
2. If there's no cycle, done.
3. Otherwise, **contract** the cycle into a single node. Incoming arcs to the cycle are rescored as score(u→v) − score(current head of v → v) + (total cycle score), i.e., the cost of breaking the cycle at $v$. Recurse on the smaller graph.
4. **Expand**: the arc chosen into the contracted node tells you where to break the cycle.

$O(n^3)$ naively, $O(n^2)$ with care. (🔎 It doesn't enforce "exactly one dependent of ROOT"; parsers add that constraint separately, e.g., by trying each root candidate.) We test it against brute-force search over all trees:

In [ ]:
def find_cycle(heads):
    # heads[0] is unused (ROOT). Returns a list of nodes forming a cycle, or None.
    for start in range(1, len(heads)):
        seen, v = [], start
        while v != 0 and v not in seen:
            seen.append(v); v = heads[v]
        if v != 0:
            return seen[seen.index(v):]
    return None

def chu_liu_edmonds(S):
    # S[h, d] = score of arc h -> d (node 0 = ROOT). Returns heads array (heads[0] = -1).
    S = S.astype(float).copy()
    n = S.shape[0]
    np.fill_diagonal(S, -np.inf)                 # no self-loops
    S[:, 0] = -np.inf                            # nothing points to ROOT
    heads = np.argmax(S, axis=0); heads[0] = -1  # 1. greedy best head for every word
    cycle = find_cycle(heads)
    if cycle is None:                            # 2. no cycle -> done
        return heads
    C = set(cycle)                               # 3. contract the cycle into one node
    rest = [i for i in range(n) if i not in C]
    idx = {v: k for k, v in enumerate(rest)}
    c = len(rest)
    cycle_score = sum(S[heads[v], v] for v in C)
    S2 = np.full((c + 1, c + 1), -np.inf)
    enter, leave = {}, {}
    for u in rest:
        for v in rest:
            S2[idx[u], idx[v]] = S[u, v]
        best_v = max(C, key=lambda v: S[u, v] - S[heads[v], v])          # best place to enter the cycle
        enter[u] = best_v
        S2[idx[u], c] = cycle_score + S[u, best_v] - S[heads[best_v], best_v]
    for v in rest:
        best_u = max(C, key=lambda u: S[u, v])                           # best cycle node to leave from
        leave[v] = best_u
        S2[c, idx[v]] = S[best_u, v]
    h2 = chu_liu_edmonds(S2)                     # recurse
    new = heads.copy()                           # 4. expand
    for v in rest:
        if v != 0:
            new[v] = leave[v] if h2[idx[v]] == c else rest[h2[idx[v]]]
    u = rest[h2[c]]
    new[enter[u]] = u                            # break the cycle where the chosen arc enters
    return new

def brute_force_best_tree(S):
    n = S.shape[0]; best, best_score = None, -np.inf
    for hs in itertools.product(range(n), repeat=n - 1):
        heads = [-1] + list(hs)
        if any(heads[i] == i for i in range(1, n)) or find_cycle(heads) is not None:
            continue
        score = sum(S[heads[i], i] for i in range(1, n))
        if score > best_score:
            best, best_score = heads, score
    return best, best_score

r = np.random.default_rng(0)
mismatch = 0
for _ in range(200):
    n = int(r.integers(2, 6))
    S = r.normal(size=(n + 1, n + 1))
    h = chu_liu_edmonds(S)
    mismatch += not np.isclose(sum(S[h[i], i] for i in range(1, n + 1)), brute_force_best_tree(S)[1])
print("Chu-Liu/Edmonds vs brute force on 200 random graphs, mismatches:", mismatch)

# The slide's toy example: ROOT The big cat sat
S = np.full((5, 5), -5.0)
S[[0, 2, 3, 4], 1] = [0.1, 0.2, 2.5, 0.1]       # head of "The"
S[[0, 1, 3, 4], 2] = [0.5, 0.3, 2.0, 0.8]       # head of "big" (numbers from the slide)
S[[0, 1, 2, 4], 3] = [0.6, 0.2, 0.4, 3.0]       # head of "cat"
S[[0, 1, 2, 3], 4] = [4.0, 0.1, 0.1, 0.5]       # head of "sat"
print("best heads for [The, big, cat, sat]:", chu_liu_edmonds(S)[1:])

### 12.2 A neural arc-factored parser
Now a small neural graph-based parser, **arc-factored**: the tree score is the sum of independent arc scores. For each candidate arc $h \to d$ we build a feature vector from the head and dependent words, their POS tags, and the (bucketed, signed) distance, then score it with a one-hidden-layer MLP:
$$\text{score}(h\to d) = u^\top\,\text{ReLU}\!\left(W[\,E^w_{h};E^t_{h};E^w_{d};E^t_{d};E^{\text{dist}}_{h-d}\,] + b\right)$$
**Training:** for each dependent $d$, a softmax over all possible heads, with cross-entropy against the gold head ("head selection", as in Dozat & Manning). **Decoding:** Chu-Liu/Edmonds over the score matrix. Because every arc is scored directly, the parser compares *spoon → read* with *spoon → picture* head-to-head, with no lookahead limit.

In [ ]:
D_G, H_G = 24, 64

def dist_bucket(delta):
    a = abs(delta)
    b = a if a <= 4 else (5 if a <= 7 else 6)
    return int(np.sign(delta) * b) + 6           # 13 buckets, 0..12

def init_graph_parser(seed=1):
    r = np.random.default_rng(seed)
    return {"Ew": r.normal(scale=0.1, size=(len(w_vocab), D_G)),
            "Ep": r.normal(scale=0.1, size=(len(p_vocab), D_G)),
            "Ed": r.normal(scale=0.1, size=(13, D_G)),
            "W": r.normal(scale=np.sqrt(2 / (5 * D_G)), size=(H_G, 5 * D_G)),
            "b": np.zeros(H_G),
            "u": r.normal(scale=np.sqrt(1 / H_G), size=H_G)}

def arc_scores(g, s):
    # Returns S of shape (n+1, n): S[h, d-1] = score(h -> d), plus a cache for backprop.
    n = len(s)
    w = np.array([W2I[ROOTTOK]] + [W2I.get(t[0], W2I[UNK]) for t in s])
    t_ = np.array([P2I[ROOTTOK]] + [P2I[t[1]] for t in s])
    hh, dd = (a.ravel() for a in np.meshgrid(np.arange(n + 1), np.arange(1, n + 1), indexing="ij"))
    ids = [("Ew", w[hh]), ("Ep", t_[hh]), ("Ew", w[dd]), ("Ep", t_[dd]),
           ("Ed", np.array([dist_bucket(h - d) for h, d in zip(hh, dd)]))]
    X = np.concatenate([g[name][i] for name, i in ids], axis=1)
    Z = X @ g["W"].T + g["b"]
    Hh = np.maximum(Z, 0)
    return (Hh @ g["u"]).reshape(n + 1, n), (X, Z, Hh, ids)

def graph_loss_and_grads(g, s):
    n = len(s)
    S, (X, Z, Hh, ids) = arc_scores(g, s)
    S[np.arange(1, n + 1), np.arange(n)] = -np.inf          # a word can't head itself
    P = softmax(S, axis=0)                                   # distribution over heads, per dependent
    gold = np.array([t[2] for t in s])
    loss = -np.log(P[gold, np.arange(n)] + 1e-12).sum()
    G = P.copy(); G[gold, np.arange(n)] -= 1; G = G.ravel()  # y_hat - y, per arc score
    grads = {"u": Hh.T @ G}
    dZ = G[:, None] * g["u"][None, :] * (Z > 0)
    grads["W"], grads["b"] = dZ.T @ X, dZ.sum(0)
    dX = dZ @ g["W"]
    for k in ("Ew", "Ep", "Ed"):
        grads[k] = np.zeros_like(g[k])
    for j, (name, i) in enumerate(ids):
        np.add.at(grads[name], i, dX[:, j * D_G:(j + 1) * D_G])
    return loss, grads

def graph_parse(g, s):
    n = len(s)
    S, _ = arc_scores(g, s)
    full = np.full((n + 1, n + 1), -np.inf); full[:, 1:] = S
    return chu_liu_edmonds(full)

gp = init_graph_parser()
opt_g, r = Adam(gp, lr=2e-3), np.random.default_rng(0)
t0 = time.time()
for ep in range(3):
    order, total, n_tok = r.permutation(len(train_sents)), 0.0, 0
    for k in range(0, len(order), 16):                       # mini-batches of 16 sentences
        acc = {key: np.zeros_like(v) for key, v in gp.items()}
        toks = 0
        for i in order[k:k + 16]:
            l, gr = graph_loss_and_grads(gp, train_sents[i])
            total += l; toks += len(train_sents[i])
            for key in acc:
                acc[key] += gr[key]
        n_tok += toks
        opt_g.step(gp, {key: v / toks for key, v in acc.items()})
    print(f"  epoch {ep + 1}: loss per token {total / n_tok:.4f}")

def graph_parse_fn(s):
    h = graph_parse(gp, s)
    return {i: int(h[i]) for i in range(1, len(s) + 1)}, {i: s[i - 1][3] for i in range(1, len(s) + 1)}  # unlabeled

uas_g, _, errors_g = evaluate(graph_parse_fn, dev_sents[:300])
print(f"trained in {time.time() - t0:.0f}s.  Graph-based UAS = {uas_g:.2%}   (transition-based: {uas:.2%})")

fixed = sum(graph_parse(gp, s)[i] == s[i - 1][2] for s, i, _ in errors)
print(f"of the {len(errors)} transition-parser errors (the adjective-PP cases), the graph parser gets {fixed} right")

(Our graph parser is unlabeled for simplicity; a second classifier over the chosen arcs predicts labels in real systems. That's an exercise below.)

### 12.3 A neural graph-based dependency parser (Dozat & Manning 2017; Dozat, Qi & Manning 2017)
This paper revived interest in graph-based dependency parsing in a neural world:
* It designed a **biaffine scoring model** for neural dependency parsing.
* It crucially uses a **neural sequence model** (a BiLSTM, next week) to build contextual representations of each word.
* Really great results! But slower than simple neural transition-based parsers ($n^2$ possible dependencies).

| Method | UAS | LAS (PTB WSJ SD 3.3) |
|---|---|---|
| Chen & Manning 2014 | 92.0 | 89.7 |
| Weiss et al. 2015 | 93.99 | 92.05 |
| Andor et al. 2016 | 94.61 | 92.79 |
| **Dozat & Manning 2017** | **95.74** | **94.08** |

**Biaffine scoring.** Run a BiLSTM over the sentence to get $r_i$ for each word. Two separate MLPs produce a "dependent" view $h_i^{(dep)}$ and a "head" view $h_j^{(head)}$. Then
$$s_{ij} = h_i^{(dep)\top}\,U\,h_j^{(head)} + h_j^{(head)\top}u$$
The bilinear term scores how well $j$ fits as the head of $i$; the linear term is a prior for how likely $j$ is to be a head at all. In matrix form all $n^2$ scores are computed at once: $S = H^{(dep)}U H^{(head)\top} + \mathbf{1}(H^{(head)}u)^\top$.

Here's a compact PyTorch implementation trained on our treebank, as a reference for Assignment-style code:

In [ ]:
# [PyTorch] BiLSTM + biaffine arc scorer (Dozat & Manning 2017), unlabeled.
import torch
import torch.nn as nn
import torch.nn.functional as F

class BiaffineParser(nn.Module):
    def __init__(self, n_words, n_pos, d=50, lstm=100, mlp=100):
        super().__init__()
        self.wemb, self.pemb = nn.Embedding(n_words, d), nn.Embedding(n_pos, d)
        self.lstm = nn.LSTM(2 * d, lstm, num_layers=2, bidirectional=True, batch_first=True)
        self.mlp_dep = nn.Sequential(nn.Linear(2 * lstm, mlp), nn.ReLU())
        self.mlp_head = nn.Sequential(nn.Linear(2 * lstm, mlp), nn.ReLU())
        self.U = nn.Parameter(torch.zeros(mlp, mlp))
        self.u = nn.Parameter(torch.zeros(mlp))

    def forward(self, words, tags):                    # (B, n+1) each; position 0 = ROOT
        r, _ = self.lstm(torch.cat([self.wemb(words), self.pemb(tags)], dim=-1))
        Hd, Hh = self.mlp_dep(r), self.mlp_head(r)     # (B, n+1, mlp)
        # S[b, i, j] = score of head j for dependent i
        return Hd @ self.U @ Hh.transpose(1, 2) + (Hh @ self.u).unsqueeze(1)

def to_tensors(s):
    w = [W2I[ROOTTOK]] + [W2I.get(t[0], W2I[UNK]) for t in s]
    p = [P2I[ROOTTOK]] + [P2I[t[1]] for t in s]
    return torch.tensor([w]), torch.tensor([p]), torch.tensor([t[2] for t in s])

torch.manual_seed(0)
bp = BiaffineParser(len(w_vocab), len(p_vocab))
opt = torch.optim.Adam(bp.parameters(), lr=2e-3)
for epoch in range(2):
    bp.train(); total = 0.0
    for i in torch.randperm(len(train_sents)).tolist():   # batch size 1 keeps the code simple (no padding)
        w, p, gold = to_tensors(train_sents[i])
        S = bp(w, p)[0, 1:]                                # (n, n+1): rows = dependents 1..n
        loss = F.cross_entropy(S, gold)                    # head selection
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"epoch {epoch + 1}: mean loss {total / len(train_sents):.4f}")

bp.eval(); correct = count = 0
with torch.no_grad():
    for s in dev_sents[:300]:
        w, p, gold = to_tensors(s)
        S = bp(w, p)[0].numpy().T                          # transpose to S[h, d] for Chu-Liu/Edmonds
        heads = chu_liu_edmonds(S)
        correct += sum(int(heads[i]) == s[i - 1][2] for i in range(1, len(s) + 1)); count += len(s)
print(f"BiLSTM-biaffine UAS: {correct / count:.2%}")

---
## 13. 🔎 Using real parsers in practice

You will rarely write a parser for production use. Mature tools: **Stanza** (Stanford, UD-trained, 70+ languages, a Dozat & Manning-style biaffine parser) and **spaCy** (fast, transition-based). Both output UD-style trees you can use for information extraction, e.g., the dependency paths of §2.4.

In [ ]:
# [Optional] needs: pip install spacy && python -m spacy download en_core_web_sm
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    for text in ["Scientists count whales from space.", "I saw the man with a telescope."]:
        doc = nlp(text)
        print(text)
        for tok in doc:
            print(f"   {tok.text:10s} --{tok.dep_:>7s}--> {tok.head.text}")
except Exception as e:
    print("Skipping spaCy demo:", e)

### 🔎 Does parsing still matter in the LLM era?
* **Explicit parsers** are still used where structure must be inspectable and cheap: information extraction pipelines (like the protein-interaction example), linguistics research, low-resource languages via UD, and grammar checking.
* **Implicit syntax:** Transformers trained only on raw text learn syntax internally. **Structural probes** (Hewitt & Manning 2019) showed that a linear transformation of BERT's hidden states recovers dependency-tree distances between words. The structure in this lecture is, in a sense, rediscovered by large models.
* The **graph-based "score all pairs" idea** reappears directly in **attention** (L8): a Transformer computes a score between every pair of words $(i, j)$, a soft, learned version of choosing a head.

---
## 14. Summary

| Concept | Takeaway |
|---|---|
| Constituency vs. dependency | Nested phrases vs. head → dependent arcs between words |
| Ambiguity | PP attachment, coordination scope, modifier scope, VP attachment; parses grow as Catalan numbers |
| Dependency tree | Single ROOT, every word has exactly one head, no cycles; UD = cross-lingual standard, CoNLL-U format |
| Projectivity | No crossing arcs; arc-standard only builds projective trees |
| Sources of information | Bilexical affinities, distance, intervening material, valency |
| Arc-standard | Stack + buffer + arcs; Shift / Left-Arc / Right-Arc; $2n$ steps, linear time |
| Oracle | Gold transition sequence from a gold tree; Right-Arc only when all dependents are attached |
| UAS / LAS | % correct heads / % correct heads and labels |
| Chen & Manning 2014 | Dense embeddings of 18 words + 18 POS + 12 labels → ReLU hidden layer → softmax over transitions |
| Greedy limitation | Decisions made with limited lookahead (we saw it on PPs with adjectives) → beam search, global training, better encoders |
| Graph-based | Score all $n^2$ arcs, decode with MST (Chu-Liu/Edmonds); biaffine + BiLSTM (Dozat & Manning 2017) is the classic strong neural parser |

### Exercises
1. Add a **labeler** to the graph parser: given a predicted arc $h\to d$, classify its label from the same arc features.
2. Implement **beam search** for the transition parser (keep the top-$k$ configurations by total log-probability). Does it fix the adjective-PP errors with $k=4$?
3. Add a **SWAP** transition (Nivre 2009) and a matching oracle, and parse *"Who did Bill buy the coffee from yesterday?"*
4. Enforce the **single-root constraint** in the graph parser: run Chu-Liu/Edmonds once per root candidate, keeping only that ROOT arc.
5. Implement **Eisner's $O(n^3)$ algorithm** for projective decoding and compare its trees with Chu-Liu/Edmonds on the dev set.
6. Download a real UD treebank (e.g., UD_English-EWT), read it with `read_conllu`, filter out non-projective trees, and train the NumPy parser on it. Expect a much lower UAS (~85–88% with this small model and no pretrained embeddings), which is a good lesson in how much harder real data is.

### References
* Nivre 2003, *An Efficient Algorithm for Projective Dependency Parsing*; Nivre 2008, *Algorithms for Deterministic Incremental Dependency Parsing*.
* Chen & Manning 2014, *A Fast and Accurate Dependency Parser using Neural Networks*.
* Weiss et al. 2015; Andor et al. 2016 (SyntaxNet, globally normalized transition-based networks).
* McDonald et al. 2005, *Non-projective Dependency Parsing using Spanning Tree Algorithms*.
* Dozat & Manning 2017, *Deep Biaffine Attention for Neural Dependency Parsing*.
* Kiperwasser & Goldberg 2016, *Simple and Accurate Dependency Parsing Using Bidirectional LSTM Feature Representations*.
* de Marneffe et al. 2021, *Universal Dependencies* (Computational Linguistics).
* Jurafsky & Martin, *Speech and Language Processing* (3rd ed.), Ch. 18 "Dependency Parsing".
* Hewitt & Manning 2019, *A Structural Probe for Finding Syntax in Word Representations*.

**Next:** Lecture 5, recurrent neural networks and language models.